# short_volshock_drawdown_drift_feature_audit_v1

## Goal

The previous notebook proved the short-volshock labels are learnable but replay still fails the `+30pp / <=15% MaxDD` gate. This notebook diagnoses the failed replay windows and turns them into a concrete reward/checkpoint objective recipe before any PPO.

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `short_volshock_bc_checkpoint_alignment_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection`, `reward`, `feature`, `checkpoint_scoring`.
- **Target windows**: `wf_2022`, `wf_2023_2024` warm-start official daily paths.
- **Acceptance**: identify shared poor-window clusters, classify gate failures, rank stable feature deltas, and write a no-promotion reward/checkpoint recipe.
- **Promotion**: blocked. This is a diagnostic handoff, not a trained actor or runtime overlay.

## External basis

- Qlib reproducible research/backtest workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validate/test separation for DRL trading: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Behavior cloning / demonstrations need downstream policy-improvement control: https://arxiv.org/abs/1704.03732
- Risk-sensitive sequential decision framing: https://arxiv.org/abs/1312.4077
- Backtest overfitting caution for repeated trials: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.oos_diagnostics import (
    aggregate_feature_differences,
    aggregate_poor_windows,
    attribute_gate_gaps,
    compute_feature_differences,
    find_poor_windows,
    load_qlib_feature_frame,
    render_gate_gap_review,
    render_oos_diagnostic_review,
    summarize_gate_gap_attribution,
)

EXPERIMENT = "short_volshock_drawdown_drift_feature_audit_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DIAG_DIR = CARD / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1"
SOURCE_SUMMARY_PATH = SOURCE_CARD / "summary.json"
SOURCE_SELECTION_PATH = SOURCE_CARD / "selection.json"
SOURCE_WARM_METRICS_PATH = SOURCE_CARD / "warm_start_official_metrics.csv"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
WINDOW_SIZE = 42
TOP_WINDOWS = 8
TOP_FEATURES = 30
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://arxiv.org/abs/1704.03732",
    "https://arxiv.org/abs/1312.4077",
    "https://doi.org/10.21314/JCF.2016.322",
]
EXTERNAL_REFERENCE_ROWS = [
    {"label": "Qlib", "url": "https://github.com/microsoft/qlib", "note": "reproducible quant workflow and artifact discipline"},
    {"label": "Qlib arXiv", "url": "https://arxiv.org/abs/2009.11189", "note": "AI-oriented quantitative investment platform"},
    {"label": "FinRL", "url": "https://github.com/AI4Finance-Foundation/FinRL", "note": "DRL trading workflow with train/validate/test separation"},
    {"label": "DQfD", "url": "https://arxiv.org/abs/1704.03732", "note": "demonstration learning still needs controlled policy improvement"},
    {"label": "Risk-sensitive RL", "url": "https://arxiv.org/abs/1312.4077", "note": "risk-sensitive sequential decision objective basis"},
    {"label": "PBO", "url": "https://doi.org/10.21314/JCF.2016.322", "note": "repeated trials require OOS and walk-forward discipline"},
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def feature_group(feature: str) -> str:
    name = str(feature).lower()
    if any(key in name for key in ["expectancy", "sharpe", "rsi"]):
        return "trend_quality"
    if any(key in name for key in ["cum_drop", "new_low", "drawdown", "downside", "semivol"]):
        return "drawdown_path"
    if any(key in name for key in ["vol", "vix", "risk", "cvar"]):
        return "volatility_regime"
    if any(key in name for key in ["negative_gap", "gap_frequency"]):
        return "gap_risk"
    if any(key in name for key in ["mom", "trend", "slope", "anchor", "gap"]):
        return "trend_momentum"
    if any(key in name for key in ["relative", "strength", "soxx", "tracking", "spread"]):
        return "relative_strength_spread"
    if any(key in name for key in ["turnover", "volume", "flow"]):
        return "liquidity_volume"
    return "other"


def compact_failure_types(values: pd.Series) -> str:
    counts = Counter(str(v) for v in values.dropna().tolist())
    return ",".join(f"{k}:{v}" for k, v in counts.most_common())

source_summary = load_json(SOURCE_SUMMARY_PATH)
source_selection = load_json(SOURCE_SELECTION_PATH)
warm_metrics = pd.read_csv(SOURCE_WARM_METRICS_PATH)
print({
    "experiment": EXPERIMENT,
    "source_decision": source_summary.get("decision"),
    "selected_candidate": source_summary.get("selected_candidate"),
    "warm_metric_rows": int(len(warm_metrics)),
})


{'experiment': 'short_volshock_drawdown_drift_feature_audit_v1', 'source_decision': 'short_volshock_bc_replay_gate_failed_no_promotion', 'selected_candidate': 'short_volshock_plain', 'warm_metric_rows': 6}


## Source Daily Paths

Use the previous card's official warm-start paths as authoritative replay evidence. The target is not to pick a promoted row; it is to explain why the closest short-volshock candidate still fails.

In [2]:
source_rows = []
for row in warm_metrics.to_dict("records"):
    if str(row.get("status", "")) != "evaluated":
        continue
    daily_path = ROOT / str(row["daily_path"])
    if not daily_path.exists():
        continue
    source_rows.append(
        {
            "label": f"{row['candidate']}__{row['fold']}",
            "candidate": row["candidate"],
            "fold": row["fold"],
            "daily_path": str(daily_path.relative_to(ROOT)),
            "relative_return": float(row["official_warm_relative_return"]),
            "max_dd": float(row["official_warm_max_dd"]),
            "avg_pos": float(row["official_warm_avg_pos"]),
            "full_gate": bool(row.get("full_gate", False)),
        }
    )
source_manifest = pd.DataFrame(source_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
source_manifest.to_csv(CARD / "source_daily_manifest.csv", index=False)
display(source_manifest)


,label,candidate,fold,daily_path,relative_return,max_dd,avg_pos,full_gate
0,base_multi_obj_plain__wf_2022,base_multi_obj_plain,wf_2022,runtime/model_selection/card/short_volshock_bc...,51.935250,36.332029,0.181179,False
1,base_multi_obj_plain__wf_2023_2024,base_multi_obj_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,33.759257,20.907231,0.253076,False
2,short_volshock_plain__wf_2022,short_volshock_plain,wf_2022,runtime/model_selection/card/short_volshock_bc...,67.624396,22.261989,0.097005,False
3,short_volshock_plain__wf_2023_2024,short_volshock_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,5.018899,16.693087,0.183114,False
4,short_volshock_weighted_obj025_cb099__wf_2022,short_volshock_weighted_obj025_cb099,wf_2022,runtime/model_selection/card/short_volshock_bc...,65.872926,24.745906,0.077558,False
5,short_volshock_weighted_obj025_cb099__wf_2023_...,short_volshock_weighted_obj025_cb099,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,8.823051,12.799187,0.144384,False


## Rolling Poor Windows And Feature Deltas

For each candidate/fold daily path, run the repo OOS diagnostic over a 42-trading-day window. Feature deltas are hypotheses only; they do not become factors without IC/signal gates.

In [3]:
windows_by_profile: dict[str, pd.DataFrame] = {}
features_by_profile: dict[str, pd.DataFrame] = {}
diag_manifest_rows: list[dict[str, Any]] = []
feature_cache: dict[tuple[str, str], pd.DataFrame] = {}

for source in source_manifest.to_dict("records"):
    label = str(source["label"])
    daily_path = ROOT / str(source["daily_path"])
    daily = pd.read_csv(daily_path, parse_dates=["datetime"])
    start = str(daily["datetime"].min().date())
    end = str(daily["datetime"].max().date())
    windows = find_poor_windows(
        daily,
        window_size=WINDOW_SIZE,
        top_n=TOP_WINDOWS,
        target_excess=TARGET_RELATIVE_RETURN,
        target_max_dd=TARGET_MAX_DD,
    )
    key = (start, end)
    if key not in feature_cache:
        feature_cache[key] = load_qlib_feature_frame(start, end)
    feature_diffs = compute_feature_differences(
        feature_cache[key],
        windows,
        daily=daily,
        top_n=TOP_FEATURES,
    )
    out_dir = DIAG_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    windows.to_csv(out_dir / "poor_windows.csv", index=False)
    feature_diffs.to_csv(out_dir / "feature_differences.csv", index=False)
    review = render_oos_diagnostic_review(
        profile=label,
        train_end="fold-local replay source",
        oos_start=start,
        oos_end=end,
        target_excess=TARGET_RELATIVE_RETURN,
        target_max_dd=TARGET_MAX_DD,
        windows=windows,
        feature_differences=feature_diffs,
        external_references=EXTERNAL_REFERENCE_ROWS,
    )
    (out_dir / "oos_diagnostic_review.md").write_text(review, encoding="utf-8")
    write_json(
        out_dir / "manifest.json",
        {
            "label": label,
            "candidate": source["candidate"],
            "fold": source["fold"],
            "daily_path": source["daily_path"],
            "oos_start": start,
            "oos_end": end,
            "window_size": WINDOW_SIZE,
            "windows": int(len(windows)),
            "features": int(len(feature_diffs)),
        },
    )
    windows_by_profile[label] = windows
    features_by_profile[label] = feature_diffs
    diag_manifest_rows.append(
        {
            "label": label,
            "candidate": source["candidate"],
            "fold": source["fold"],
            "diagnostic_dir": str(out_dir.relative_to(ROOT)),
            "window_count": int(len(windows)),
            "feature_count": int(len(feature_diffs)),
            "worst_bad_score": float(windows["bad_score"].max()) if not windows.empty else 0.0,
        }
    )

diag_manifest = pd.DataFrame(diag_manifest_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
diag_manifest.to_csv(CARD / "diagnostic_manifest.csv", index=False)
display(diag_manifest)


[49824:MainThread](2026-05-26 20:43:53,446) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[49824:MainThread](2026-05-26 20:43:54,133) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[49824:MainThread](2026-05-26 20:43:54,134) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[49824:MainThread](2026-05-26 20:43:56,728) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[49824:MainThread](2026-05-26 20:43:56,729) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[49824:MainThread](2026-05-26 20:43:56,972) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[49824:MainThread](2026-05-26 20:43:59,622) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[49824:MainThread](2026-05-26 20:43:59,624) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[49824:MainThread](2026-05-26 20:43:59,625) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | Init data Done


,label,candidate,fold,diagnostic_dir,window_count,feature_count,worst_bad_score
0,base_multi_obj_plain__wf_2022,base_multi_obj_plain,wf_2022,runtime/model_selection/card/short_volshock_dr...,8,30,117.743074
1,base_multi_obj_plain__wf_2023_2024,base_multi_obj_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_dr...,8,30,114.428888
2,short_volshock_plain__wf_2022,short_volshock_plain,wf_2022,runtime/model_selection/card/short_volshock_dr...,8,30,118.208430
3,short_volshock_plain__wf_2023_2024,short_volshock_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_dr...,8,30,129.519264
4,short_volshock_weighted_obj025_cb099__wf_2022,short_volshock_weighted_obj025_cb099,wf_2022,runtime/model_selection/card/short_volshock_dr...,8,30,119.649494
5,short_volshock_weighted_obj025_cb099__wf_2023_...,short_volshock_weighted_obj025_cb099,wf_2023_2024,runtime/model_selection/card/short_volshock_dr...,8,30,129.639564


## Aggregate Failure Modes

Aggregate the rolling windows and feature deltas across all replay candidates, then classify promotion gate misses from the official warm-start metrics.

In [4]:
clusters = aggregate_poor_windows(
    windows_by_profile,
    target_excess=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
feature_clusters = aggregate_feature_differences(features_by_profile, top_n=TOP_FEATURES)
feature_clusters["feature_group"] = feature_clusters["feature"].map(feature_group) if not feature_clusters.empty else []
feature_group_summary = (
    feature_clusters.groupby("feature_group", dropna=False)
    .agg(
        feature_count=("feature", "count"),
        mean_stability_score=("stability_score", "mean"),
        max_stability_score=("stability_score", "max"),
        mean_abs_delta_z=("mean_abs_delta_z", "mean"),
        top_features=("feature", lambda s: ",".join(list(s.head(5)))),
    )
    .reset_index()
    .sort_values(["max_stability_score", "feature_count"], ascending=[False, False])
    if not feature_clusters.empty
    else pd.DataFrame()
)

metrics_by_profile: dict[str, dict[str, dict[str, float]]] = {}
for candidate, group in warm_metrics[warm_metrics["status"].eq("evaluated")].groupby("candidate", sort=True):
    metrics_by_profile[str(candidate)] = {}
    for row in group.to_dict("records"):
        metrics_by_profile[str(candidate)][str(row["fold"])] = {
            "return": float(row["official_warm_return"]),
            "buy_and_hold_return": float(row["official_warm_buyhold_return"]),
            "relative_return": float(row["official_warm_relative_return"]),
            "max_dd": float(row["official_warm_max_dd"]),
            "avg_pos": float(row["official_warm_avg_pos"]),
        }

gate_gaps = attribute_gate_gaps(
    metrics_by_profile,
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
gate_summary = summarize_gate_gap_attribution(gate_gaps)
review = render_gate_gap_review(gate_gaps, gate_summary, external_references=EXTERNAL_REFERENCE_ROWS)

clusters.to_csv(CARD / "poor_window_clusters.csv", index=False)
feature_clusters.to_csv(CARD / "feature_delta_clusters.csv", index=False)
feature_group_summary.to_csv(CARD / "feature_group_summary.csv", index=False)
gate_gaps.to_csv(CARD / "gate_gap_attribution.csv", index=False)
gate_summary.to_csv(CARD / "gate_gap_profile_summary.csv", index=False)
(CARD / "gate_gap_review.md").write_text(review, encoding="utf-8")

display(clusters.head(12))
display(feature_clusters.head(20))
display(feature_group_summary)
display(gate_gaps)
display(gate_summary)


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C004,2024-04-18,2024-06-21,3,11,"base_multi_obj_plain__wf_2023_2024,short_volsh...",-73.865553,-99.639564,3.173384,4.848972,0.216986,103.865553,129.639564,underparticipation,reentry_participation_or_behavior_clone
1,C002,2022-10-11,2022-12-16,3,19,"base_multi_obj_plain__wf_2022,short_volshock_p...",-63.674948,-89.649494,7.235904,10.787106,0.192848,93.674948,119.649494,underparticipation,reentry_participation_or_behavior_clone
2,C003,2024-01-03,2024-03-12,3,13,"base_multi_obj_plain__wf_2023_2024,short_volsh...",-66.540519,-82.688936,5.661665,8.238349,0.331419,96.540519,112.688936,underparticipation,reentry_participation_or_behavior_clone
3,C001,2022-06-16,2022-08-17,3,5,"base_multi_obj_plain__wf_2022,short_volshock_p...",-47.958779,-54.269808,7.411798,9.467220,0.152457,77.958779,84.269808,underparticipation,reentry_participation_or_behavior_clone


,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score,feature_group
0,expectancy_proxy_21,6,1.252761,1.252761,1.0,-0.241092,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.437761,trend_quality
1,vix_percentile_252,6,-1.195318,1.195318,1.0,0.186677,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.325981,volatility_regime
2,sharpe_proxy_21,6,1.194851,1.194851,1.0,-0.201593,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.325073,trend_quality
3,regime_risk,6,-1.128463,1.128463,1.0,0.181282,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.195887,volatility_regime
4,mom_21,6,1.068319,1.068319,1.0,-0.205605,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.078853,trend_momentum
5,relative_strength_21,6,1.064950,1.064950,1.0,-0.203585,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.072297,relative_strength_spread
6,rsi_14,6,1.035324,1.035324,1.0,-0.207902,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",2.014648,trend_quality
7,negative_gap_frequency_21,3,-1.392979,1.392979,1.0,0.094926,"base_multi_obj_plain__wf_2023_2024,short_volsh...",1.931079,gap_risk
8,trend_slope_21,6,0.952631,0.952631,1.0,0.041160,"base_multi_obj_plain__wf_2022,base_multi_obj_p...",1.853735,trend_momentum
9,drawdown_63,3,1.330989,1.330989,1.0,-0.165453,"base_multi_obj_plain__wf_2023_2024,short_volsh...",1.845142,drawdown_path


,feature_group,feature_count,mean_stability_score,max_stability_score,mean_abs_delta_z,top_features
5,trend_quality,4,2.013453,2.437761,1.048817,"expectancy_proxy_21,sharpe_proxy_21,rsi_14,rsi_6"
6,volatility_regime,9,1.492952,2.325981,0.933188,"vix_percentile_252,regime_risk,vix_zscore,real..."
4,trend_momentum,7,1.611204,2.078853,0.929748,"mom_21,trend_slope_21,trend_gap_21,mom_63,anch..."
3,relative_strength_spread,1,2.072297,2.072297,1.064950,relative_strength_21
1,gap_risk,1,1.931079,1.931079,1.392979,negative_gap_frequency_21
0,drawdown_path,7,1.660658,1.845142,0.996559,"drawdown_63,cum_drop_30,downside_semivol_21,dr..."
2,other,1,1.192759,1.192759,0.860394,realized_kurtosis_21


,profile,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,relative_gap,max_dd_gap,relative_pass,maxdd_pass,gate_pass,failure_type,severity_score,primary_optimization_path
0,base_multi_obj_plain,wf_2022,-34.652822,-86.588072,51.935250,NaN,36.332029,0.181179,0.000000,21.332029,True,False,False,drawdown_control,213.320291,drawdown_controller_or_sequence_risk_feature
1,short_volshock_weighted_obj025_cb099,wf_2022,-20.715146,-86.588072,65.872926,NaN,24.745906,0.077558,0.000000,9.745906,True,False,False,drawdown_control,97.459064,drawdown_controller_or_sequence_risk_feature
2,short_volshock_plain,wf_2022,-18.963676,-86.588072,67.624396,NaN,22.261989,0.097005,0.000000,7.261989,True,False,False,drawdown_control,72.619890,drawdown_controller_or_sequence_risk_feature
3,base_multi_obj_plain,wf_2023_2024,33.402622,-0.356635,33.759257,NaN,20.907231,0.253076,0.000000,5.907231,True,False,False,drawdown_control,59.072310,drawdown_controller_or_sequence_risk_feature
4,short_volshock_plain,wf_2023_2024,4.662264,-0.356635,5.018899,NaN,16.693087,0.183114,24.981101,1.693087,False,False,False,joint_tradeoff,41.911976,walk_forward_weighted_multi_objective_selection
5,short_volshock_weighted_obj025_cb099,wf_2023_2024,8.466416,-0.356635,8.823051,NaN,12.799187,0.144384,21.176949,0.000000,False,True,False,selection_bias,21.176949,checkpoint_or_model_selection


,profile,window_count,gate_pass_count,failure_count,dominant_failure_type,worst_window,max_severity_score,recommended_next_mechanism
0,base_multi_obj_plain,2,0,2,drawdown_control,wf_2022,213.320291,drawdown_controller_or_sequence_risk_feature
1,short_volshock_weighted_obj025_cb099,2,0,2,drawdown_control,wf_2022,97.459064,drawdown_controller_or_sequence_risk_feature
2,short_volshock_plain,2,0,2,drawdown_control,wf_2022,72.619890,drawdown_controller_or_sequence_risk_feature


## Reward / Checkpoint Recipe

This cell converts the diagnostic into a constrained next-step recipe. The recipe is intentionally not a profile and not a promotion; it is a target for the next reward/checkpoint screen.

In [5]:
selected_candidate = str(source_summary.get("selected_candidate", "short_volshock_plain"))
selected_gaps = gate_gaps[gate_gaps["profile"].eq(selected_candidate)].copy()
selected_failures = compact_failure_types(selected_gaps["failure_type"]) if not selected_gaps.empty else ""
dominant_failure = (
    str(gate_summary[gate_summary["profile"].eq(selected_candidate)]["dominant_failure_type"].iloc[0])
    if not gate_summary[gate_summary["profile"].eq(selected_candidate)].empty
    else "unknown"
)
selected_clusters = clusters[clusters["profiles"].astype(str).str.contains(selected_candidate, regex=False)].copy()
shared_cluster_count = int((clusters["profile_count"] >= 2).sum()) if not clusters.empty else 0
top_group = str(feature_group_summary.iloc[0]["feature_group"]) if not feature_group_summary.empty else "unknown"
top_features = feature_clusters.head(10)["feature"].tolist() if not feature_clusters.empty else []

wf2022 = selected_gaps[selected_gaps["window"].eq("wf_2022")]
wf2324 = selected_gaps[selected_gaps["window"].eq("wf_2023_2024")]
wf2022_type = str(wf2022["failure_type"].iloc[0]) if not wf2022.empty else "missing"
wf2324_type = str(wf2324["failure_type"].iloc[0]) if not wf2324.empty else "missing"

if {wf2022_type, wf2324_type} & {"drawdown_control", "joint_tradeoff", "near_miss_drawdown"}:
    decision = "short_volshock_drawdown_drift_joint_objective_needed_no_promotion"
    next_mechanism = "screen drawdown-drift reward/checkpoint penalty with trend-preservation term before PPO"
else:
    decision = "short_volshock_failure_mode_unclear_no_promotion"
    next_mechanism = "collect additional fixed-actor or factor diagnostics before PPO"

recipe = {
    "name": "short_volshock_drawdown_drift_joint_objective_recipe_v1",
    "decision": decision,
    "promotion_allowed": False,
    "ppo_allowed_next": False,
    "selected_candidate": selected_candidate,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "failure_summary": {
        "dominant_failure": dominant_failure,
        "selected_failures": selected_failures,
        "wf_2022_failure": wf2022_type,
        "wf_2023_2024_failure": wf2324_type,
        "shared_poor_window_cluster_count": shared_cluster_count,
        "top_feature_group": top_group,
        "top_features": top_features,
    },
    "objective_terms": [
        {
            "term": "drawdown_drift_penalty",
            "scope": "fold-local replay and checkpoint scoring",
            "definition": "penalize candidate MaxDD above 15% and any PPO/checkpoint MaxDD drift above the BC or base reference",
            "weight_hint": "start with gate_loss scale: max_dd_gap * 10",
        },
        {
            "term": "short_volshock_state_deleverage_consistency",
            "scope": "target/reward rows where short_volshock_target_state is true",
            "definition": "reward low position only during high short-volshock/de-risk states; avoid global low-exposure drift",
            "weight_hint": "state-conditional, not global exposure penalty",
        },
        {
            "term": "trend_preservation_floor",
            "scope": "non-short-volshock rows with positive trend/relative-strength features",
            "definition": "penalize underparticipation in trend windows so wf_2023_2024 relative return does not collapse",
            "weight_hint": "activate only outside drawdown-risk state",
        },
        {
            "term": "checkpoint_objective_loss",
            "scope": "model selection before PPO lengthening",
            "definition": "rank actors by alignment + gate loss + drawdown drift, not validation score alone",
            "weight_hint": "reuse checkpoint_objective_scores from previous card",
        },
    ],
    "feature_hypotheses": [
        {
            "group": row["feature_group"],
            "top_features": str(row["top_features"]).split(",") if pd.notna(row["top_features"]) else [],
            "mean_abs_delta_z": float(row["mean_abs_delta_z"]),
            "max_stability_score": float(row["max_stability_score"]),
        }
        for row in feature_group_summary.head(5).to_dict("records")
    ] if not feature_group_summary.empty else [],
    "next_mechanism": next_mechanism,
    "blocked_paths": [
        "do_not_extend_same_short_volshock_bc_actor_into_longer_ppo",
        "do_not_promote_runtime_overlay_from_this_diagnostic",
        "do_not_use_feature_deltas_as_live_factors_without_ic_and_walkforward_signal_tests",
    ],
}
write_json(CARD / "reward_checkpoint_recipe.json", recipe)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": "sparse_release_bc_ppo_screen_v1",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": ["short_volshock_bc_checkpoint_alignment_v1"],
    "closest_historical_success_case": "short_volshock target generation and BC alignment learned the short-volshock target slice with high accuracy",
    "closest_historical_failure_case": "short_volshock BC replay failed the walk-forward gate because wf_2022 MaxDD remained high and wf_2023_2024 relative return collapsed",
    "duplicate_check": "This run does not retrain or retune BC; it aggregates poor windows/features and writes a reward/checkpoint recipe.",
    "changed_layer": ["model_selection", "reward", "feature", "checkpoint_scoring"],
    "selected_candidate": selected_candidate,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "window_size": WINDOW_SIZE,
    "dominant_failure": dominant_failure,
    "top_feature_group": top_group,
    "promotion_allowed": False,
    "ppo_allowed_next": False,
    "training_allowed_from_this_card": False,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "source_daily_manifest.csv",
        "diagnostic_manifest.csv",
        "diagnostics/*/poor_windows.csv",
        "diagnostics/*/feature_differences.csv",
        "poor_window_clusters.csv",
        "feature_delta_clusters.csv",
        "feature_group_summary.csv",
        "gate_gap_attribution.csv",
        "gate_gap_profile_summary.csv",
        "gate_gap_review.md",
        "reward_checkpoint_recipe.json",
        "selection.json",
        "summary.json",
    ],
}

conclusion = [
    f"Selected source `{selected_candidate}` has gate failures `{selected_failures}`; decision `{decision}`.",
    f"`wf_2022` failure type is `{wf2022_type}` and `wf_2023_2024` failure type is `{wf2324_type}`.",
    f"Shared poor-window clusters: {shared_cluster_count}; top feature group: `{top_group}`.",
    "The next valid optimization is a drawdown-drift reward/checkpoint screen with trend preservation; not longer PPO from the current BC actor.",
    "Feature deltas are mechanism hypotheses only and must pass factor IC / single-signal / walk-forward checks before promotion.",
]
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "selected_candidate": selected_candidate,
    "promotion_allowed": False,
    "ppo_allowed_next": False,
    "training_allowed_from_this_card": False,
    "dominant_failure": dominant_failure,
    "selected_failures": selected_failures,
    "wf_2022_failure": wf2022_type,
    "wf_2023_2024_failure": wf2324_type,
    "shared_poor_window_cluster_count": shared_cluster_count,
    "top_feature_group": top_group,
    "top_features": top_features,
    "recipe": recipe,
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))
display(pd.DataFrame([recipe["failure_summary"]]))


{
  "experiment": "short_volshock_drawdown_drift_feature_audit_v1",
  "created_at": "2026-05-26T12:44:00.840743+00:00",
  "decision": "short_volshock_drawdown_drift_joint_objective_needed_no_promotion",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "short_volshock_bc_checkpoint_alignment_v1"
  ],
  "closest_historical_success_case": "short_volshock target generation and BC alignment learned the short-volshock target slice with high accuracy",
  "closest_historical_failure_case": "short_volshock BC replay failed the walk-forward gate because wf_2022 MaxDD remained high and wf_2023_2024 relative return collapsed",
  "duplicate_check": "This run does not retrain or retune BC; it aggregates poor windows/features and writes a reward/checkpoint recipe.",
  "changed_layer": [
    "model_selection",
    "reward",
    "feature",
    "checkpoint_scoring"
  ],
  "selected

,dominant_failure,selected_failures,wf_2022_failure,wf_2023_2024_failure,shared_poor_window_cluster_count,top_feature_group,top_features
0,drawdown_control,"drawdown_control:1,joint_tradeoff:1",drawdown_control,joint_tradeoff,4,trend_quality,"[expectancy_proxy_21, vix_percentile_252, shar..."


## Runtime Conclusion

The persisted conclusion below is the single-run source of truth. Sidecar CSV/JSON files support replay and indexing.

In [6]:
selection = load_json(CARD / "selection.json")
summary = load_json(CARD / "summary.json")
lines = "\n".join(f"- {line}" for line in summary["conclusion"])
md = f"""
Decision: `{selection['decision']}`. Promotion is blocked.

Next mechanism: `{selection['next_mechanism']}`.

{lines}
"""
display(Markdown(md))



Decision: `short_volshock_drawdown_drift_joint_objective_needed_no_promotion`. Promotion is blocked.

Next mechanism: `screen drawdown-drift reward/checkpoint penalty with trend-preservation term before PPO`.

- Selected source `short_volshock_plain` has gate failures `drawdown_control:1,joint_tradeoff:1`; decision `short_volshock_drawdown_drift_joint_objective_needed_no_promotion`.
- `wf_2022` failure type is `drawdown_control` and `wf_2023_2024` failure type is `joint_tradeoff`.
- Shared poor-window clusters: 4; top feature group: `trend_quality`.
- The next valid optimization is a drawdown-drift reward/checkpoint screen with trend preservation; not longer PPO from the current BC actor.
- Feature deltas are mechanism hypotheses only and must pass factor IC / single-signal / walk-forward checks before promotion.
